# Create AnnData from Xenium output folders

**Pinned Environment:** [`envs/sc-spatial.yaml`](../../envs/sc-spatial.yaml)

In [1]:
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import math
import session_info

In [2]:
plt.rcParams['figure.dpi'] = 150

## Set paths

In [3]:
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_DIR, DATA_DIR

data_input_folder = DATA_DIR
h5ad_dir = BASE_DIR / 'data/h5ad/export_01/01a_raw/output_folder'

# Create directories
BASE_DIR.mkdir(parents=True, exist_ok=True)
h5ad_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
sample_id_mapping = {
    "output-XETG00123__0052281__TMA00303__20251108__004334": {"region": "TMA00303", "slide": "0052281"},
    "output-XETG00123__0052334__TMA00304__20251108__004334": {"region": "TMA00304", "slide": "0052334"},
    "output-XETG00123__0052281__TMA00305__20251108__004334": {"region": "TMA00305", "slide": "0052281"},
    "output-XETG00123__0052334__TMA00306__20251108__004334": {"region": "TMA00306", "slide": "0052334"},
    "output-XETG00123__0052046__TMA00307__20251119__221941": {"region": "TMA00307", "slide": "0052046"},
    "output-XETG00123__0052399__TMA00308__20251119__221941": {"region": "TMA00308", "slide": "0052399"},
    "output-XETG00123__0052046__TMA00311__20251119__221941": {"region": "TMA00311", "slide": "0052046"},
    "output-XETG00123__0052399__TMA00312__20251119__221941": {"region": "TMA00312", "slide": "0052399"},
    "output-XETG00123__0051689__TMA00309__20251203__231448": {"region": "TMA00309", "slide": "0051689"},
    "output-XETG00123__0051805__TMA00310__20251203__231448": {"region": "TMA00310", "slide": "0051805"},
    "output-XETG00123__0051689__TMA00313__20251203__231448": {"region": "TMA00313", "slide": "0051689"},
    "output-XETG00123__0051805__TMA00314__20251203__231448": {"region": "TMA00314", "slide": "0051805"},
}

## Functions

In [ ]:
def h5_to_adata(directory, sample_id):
    # File names    
    h5_file = os.path.join(directory, 'cell_feature_matrix.h5')
    cells_file = os.path.join(directory, 'cells.csv.gz')

    # h5 to adata
    adata = sc.read_10x_h5(
        filename=h5_file
    )

    # cells file
    df = pd.read_csv(
        cells_file, 
        compression='gzip'
    )

    # Set index
    df.set_index(adata.obs_names, inplace=True)
    adata.obs = df.copy()

    # x, y coordinates
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

    # Add sample metadata
    adata.obs['sample_id'] = sample_id

    return adata

## Create sample dictionary

In [ ]:
directory_to_sample = {
    os.path.join(data_input_folder, subdir): f"{subdir}"
    for subdir in os.listdir(data_input_folder) if os.path.isdir(os.path.join(data_input_folder, subdir))
    if os.path.isdir(os.path.join(data_input_folder, subdir)) and not subdir.startswith(".")  # Excludes hidden folders like .ipynb_checkpoints
}

print(directory_to_sample)

## Create anndata

In [ ]:
# Loop through each key-value pair in the dictionary
adata_list = []
for directory, output_id in directory_to_sample.items():
    print(f"Processing: {directory} -> Run ID: {output_id}")
    adata = h5_to_adata(directory, output_id)
    adata.obs["output_id"] = output_id  
    adata_list.append(adata)

In [ ]:
for adata in adata_list:
    adata.obs["sample_id"] = adata.obs["output_id"].map(lambda x: sample_id_mapping[x]["region"])
    adata.obs["region_id"] = adata.obs["sample_id"]
    adata.obs["slide_id"]  = adata.obs["output_id"].map(lambda x: sample_id_mapping[x]["slide"])

    print(adata.obs[["output_id", "sample_id", "region_id", "slide_id"]].head(1))

## QC

In [ ]:
adata_list

#### Fix order

In [ ]:
adata_list = sorted(adata_list, key=lambda ad: ad.obs["sample_id"].iloc[0])

#### Visualize

In [ ]:
for adata in adata_list:
    sample_id = adata.obs['sample_id'].iloc[0]
    print(sample_id)

    # Calculate QC
    sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

    # Number of cells
    cell_num = adata.obs.shape[0]
    print(cell_num, 'cells')

    # Visualize QC
    fig, axs = plt.subplots(1, 2, figsize=(15, 4))

    # Plot the first histogram: Total Counts
    counts_median = adata.obs["total_counts"].median()  
    axs[0].hist(adata.obs["total_counts"], bins=math.ceil(cell_num / 50), edgecolor='black')
    axs[0].set_title(sample_id + ", " + "counts/cell")
    axs[0].set_xlim(0, 1500)
    axs[0].set_ylim(0, 3000)
    axs[0].set_xlabel('Total Counts')
    axs[0].set_ylabel('Frequency')
    axs[0].axvline(counts_median, color='red', linestyle='--', label=f'Median: {counts_median:.2f}')  # Add vertical line
    axs[0].legend()  # Add legend for the median line

    # Plot the second histogram: Genes by Counts
    genes_median = adata.obs["n_genes_by_counts"].median()  
    axs[1].hist(adata.obs["n_genes_by_counts"], bins=math.ceil(cell_num / 50), edgecolor='black')
    axs[1].set_title(sample_id + ", " + "genes/cell")
    axs[1].set_xlim(0, 300)
    axs[1].set_xlabel('Unique Transcripts (genes)')
    axs[1].set_ylabel('Frequency')
    axs[1].axvline(genes_median, color='red', linestyle='--', label=f'Median: {genes_median:.2f}')  # Add vertical line
    axs[1].legend()  # Add legend for the median line


## Genes of interest

In [ ]:
genes = adata.var
genes.head()

## Export Anndata

In [ ]:
# export 01

for i, adata in enumerate(adata_list):
    sample_name = adata.obs["sample_id"][0] if "sample_id" in adata.obs.columns else f"sample_{i+1}"
    output_file = os.path.join(h5ad_dir, f"{sample_name}.h5ad")
    
    adata.write(output_file)
    print(f"Saved: {output_file}")